# 1.1 NBA Arena Locations and Historical Team Identification

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("../data/raw/nba_all_regular_seasons.csv")

games_all = pd.read_csv(
    data_path,
    dtype={"GAME_ID": str, "SEASON": str}
)

teams_summary = (
    games_all
    .sort_values("SEASON")
    .groupby("TEAM_ID", as_index=False)
    .agg(
        team_abbreviation=("TEAM_ABBREVIATION", "last"),
        team_name=("TEAM_NAME", "last"),
        first_season=("SEASON", "min"),
        last_season=("SEASON", "max"),
        team_games=("GAME_ID", "count")
    )
    .sort_values("team_abbreviation")
    .reset_index(drop=True)
)

print(f"Total team-game records: {len(games_all):,}")
print(f"Unique seasons: {games_all['SEASON'].nunique()}")
print(f"Unique games: {games_all['GAME_ID'].nunique():,}")
print(f"Unique team IDs: {games_all['TEAM_ID'].nunique()}")
print(f"Unique team records: {len(teams_summary)}")

display(teams_summary)


Total team-game records: 26,418
Unique seasons: 11
Unique games: 13,209
Unique team IDs: 30
Unique team records: 30


,TEAM_ID,team_abbreviation,team_name,first_season,last_season,team_games
0,1610612737,ATL,Atlanta Hawks,2014-15,2024-25,877
1,1610612751,BKN,Brooklyn Nets,2014-15,2024-25,882
2,1610612738,BOS,Boston Celtics,2014-15,2024-25,882
3,1610612766,CHA,Charlotte Hornets,2014-15,2024-25,875
4,1610612741,CHI,Chicago Bulls,2014-15,2024-25,875
5,1610612739,CLE,Cleveland Cavaliers,2014-15,2024-25,875
6,1610612742,DAL,Dallas Mavericks,2014-15,2024-25,885
7,1610612743,DEN,Denver Nuggets,2014-15,2024-25,883
8,1610612765,DET,Detroit Pistons,2014-15,2024-25,876
9,1610612744,GSW,Golden State Warriors,2014-15,2024-25,875


# 1.2 NBA Team Home Cities and Historical Changes

In [2]:
team_cities = {
    "ATL": ("Atlanta", "Georgia", "USA"),
    "BOS": ("Boston", "Massachusetts", "USA"),
    "BKN": ("Brooklyn", "New York", "USA"),
    "CHA": ("Charlotte", "North Carolina", "USA"),
    "CHI": ("Chicago", "Illinois", "USA"),
    "CLE": ("Cleveland", "Ohio", "USA"),
    "DAL": ("Dallas", "Texas", "USA"),
    "DEN": ("Denver", "Colorado", "USA"),
    "DET": ("Detroit", "Michigan", "USA"),
    "GSW": ("San Francisco", "California", "USA"),
    "HOU": ("Houston", "Texas", "USA"),
    "IND": ("Indianapolis", "Indiana", "USA"),
    "LAC": ("Los Angeles", "California", "USA"),
    "LAL": ("Los Angeles", "California", "USA"),
    "MEM": ("Memphis", "Tennessee", "USA"),
    "MIA": ("Miami", "Florida", "USA"),
    "MIL": ("Milwaukee", "Wisconsin", "USA"),
    "MIN": ("Minneapolis", "Minnesota", "USA"),
    "NOP": ("New Orleans", "Louisiana", "USA"),
    "NYK": ("New York", "New York", "USA"),
    "OKC": ("Oklahoma City", "Oklahoma", "USA"),
    "ORL": ("Orlando", "Florida", "USA"),
    "PHI": ("Philadelphia", "Pennsylvania", "USA"),
    "PHX": ("Phoenix", "Arizona", "USA"),
    "POR": ("Portland", "Oregon", "USA"),
    "SAC": ("Sacramento", "California", "USA"),
    "SAS": ("San Antonio", "Texas", "USA"),
    "TOR": ("Toronto", "Ontario", "Canada"),
    "UTA": ("Salt Lake City", "Utah", "USA"),
    "WAS": ("Washington", "District of Columbia", "USA")
}

city_mapping = pd.DataFrame(
    [
        {
            "team_abbreviation": abbreviation,
            "city": location[0],
            "state_province": location[1],
            "country": location[2]
        }
        for abbreviation, location in team_cities.items()
    ]
)

city_mapping = teams_summary.merge(
    city_mapping,
    on="team_abbreviation",
    how="left",
    validate="one_to_one"
)

city_mapping.loc[
    (city_mapping["team_abbreviation"] == "GSW")
    & (city_mapping["first_season"] < "2019-20"),
    "city"
] = "San Francisco"

city_mapping = city_mapping.sort_values(
    "team_abbreviation"
).reset_index(drop=True)

print(f"Teams mapped: {city_mapping['city'].notna().sum()}")
print(f"Missing mappings: {city_mapping['city'].isna().sum()}")

display(city_mapping)


Teams mapped: 30
Missing mappings: 0


,TEAM_ID,team_abbreviation,team_name,first_season,last_season,team_games,city,state_province,country
0,1610612737,ATL,Atlanta Hawks,2014-15,2024-25,877,Atlanta,Georgia,USA
1,1610612751,BKN,Brooklyn Nets,2014-15,2024-25,882,Brooklyn,New York,USA
2,1610612738,BOS,Boston Celtics,2014-15,2024-25,882,Boston,Massachusetts,USA
3,1610612766,CHA,Charlotte Hornets,2014-15,2024-25,875,Charlotte,North Carolina,USA
4,1610612741,CHI,Chicago Bulls,2014-15,2024-25,875,Chicago,Illinois,USA
5,1610612739,CLE,Cleveland Cavaliers,2014-15,2024-25,875,Cleveland,Ohio,USA
6,1610612742,DAL,Dallas Mavericks,2014-15,2024-25,885,Dallas,Texas,USA
7,1610612743,DEN,Denver Nuggets,2014-15,2024-25,883,Denver,Colorado,USA
8,1610612765,DET,Detroit Pistons,2014-15,2024-25,876,Detroit,Michigan,USA
9,1610612744,GSW,Golden State Warriors,2014-15,2024-25,875,San Francisco,California,USA


# 1.3 Season-Specific NBA Home City Mapping

In [3]:
seasons = sorted(games_all["SEASON"].unique())

season_city_mapping = (
    city_mapping.assign(_key=1)
    .merge(
        pd.DataFrame({"SEASON": seasons, "_key": 1}),
        on="_key"
    )
    .drop(columns="_key")
)

season_city_mapping.loc[
    (season_city_mapping["team_abbreviation"] == "GSW")
    & (season_city_mapping["SEASON"] < "2019-20"),
    "city"
] = "Oakland"

season_city_mapping = (
    season_city_mapping[
        [
            "TEAM_ID",
            "team_abbreviation",
            "SEASON",
            "city",
            "state_province",
            "country"
        ]
    ]
    .sort_values(["SEASON", "team_abbreviation"])
    .reset_index(drop=True)
)

print(f"Total mappings: {len(season_city_mapping)}")
print(f"Unique teams: {season_city_mapping['TEAM_ID'].nunique()}")
print(f"Unique seasons: {season_city_mapping['SEASON'].nunique()}")
print(f"Missing cities: {season_city_mapping['city'].isna().sum()}")

display(
    season_city_mapping[
        season_city_mapping["team_abbreviation"] == "GSW"
    ]
)


Total mappings: 330
Unique teams: 30
Unique seasons: 11
Missing cities: 0


,TEAM_ID,team_abbreviation,SEASON,city,state_province,country
9,1610612744,GSW,2014-15,Oakland,California,USA
39,1610612744,GSW,2015-16,Oakland,California,USA
69,1610612744,GSW,2016-17,Oakland,California,USA
99,1610612744,GSW,2017-18,Oakland,California,USA
129,1610612744,GSW,2018-19,Oakland,California,USA
159,1610612744,GSW,2019-20,San Francisco,California,USA
189,1610612744,GSW,2020-21,San Francisco,California,USA
219,1610612744,GSW,2021-22,San Francisco,California,USA
249,1610612744,GSW,2022-23,San Francisco,California,USA
279,1610612744,GSW,2023-24,San Francisco,California,USA


# 1.4.1 Game City Assignment

In [4]:
games_with_city = games_all.copy()

games_with_city["is_home"] = games_with_city["MATCHUP"].str.contains(
    "vs.", regex=False
)

home_games = games_with_city.loc[
    games_with_city["is_home"],
    ["GAME_ID", "SEASON", "TEAM_ID"]
].rename(columns={"TEAM_ID": "HOME_TEAM_ID"})

home_games = home_games.merge(
    season_city_mapping[
        ["TEAM_ID", "SEASON", "city", "state_province", "country"]
    ],
    left_on=["HOME_TEAM_ID", "SEASON"],
    right_on=["TEAM_ID", "SEASON"],
    how="left",
    validate="many_to_one"
).drop(columns="TEAM_ID")

home_games = home_games.rename(
    columns={
        "city": "GAME_CITY",
        "state_province": "GAME_STATE_PROVINCE",
        "country": "GAME_COUNTRY"
    }
)

games_with_city = games_with_city.merge(
    home_games[
        ["GAME_ID", "GAME_CITY", "GAME_STATE_PROVINCE", "GAME_COUNTRY"]
    ],
    on="GAME_ID",
    how="left",
    validate="many_to_one"
)

games_with_city["LOCATION_STATUS"] = "home_city_assumption"

print(f"Total team-game records: {len(games_with_city):,}")
print(f"Unique games: {games_with_city['GAME_ID'].nunique():,}")
print(f"Missing game cities: {games_with_city['GAME_CITY'].isna().sum():,}")

display(
    games_with_city[
        [
            "GAME_ID",
            "GAME_DATE",
            "TEAM_ABBREVIATION",
            "MATCHUP",
            "GAME_CITY",
            "LOCATION_STATUS"
        ]
    ].head(10)
)


Total team-game records: 26,418
Unique games: 13,209
Missing game cities: 10


,GAME_ID,GAME_DATE,TEAM_ABBREVIATION,MATCHUP,GAME_CITY,LOCATION_STATUS
0,0021401222,2015-04-15,OKC,OKC @ MIN,Minneapolis,home_city_assumption
1,0021401217,2015-04-15,ATL,ATL @ CHI,Chicago,home_city_assumption
2,0021401229,2015-04-15,DEN,DEN @ GSW,Oakland,home_city_assumption
3,0021401225,2015-04-15,WAS,WAS @ CLE,Cleveland,home_city_assumption
4,0021401228,2015-04-15,CHA,CHA @ TOR,Toronto,home_city_assumption
5,0021401229,2015-04-15,GSW,GSW vs. DEN,Oakland,home_city_assumption
6,0021401221,2015-04-15,BOS,BOS @ MIL,Milwaukee,home_city_assumption
7,0021401217,2015-04-15,CHI,CHI vs. ATL,Chicago,home_city_assumption
8,0021401223,2015-04-15,SAS,SAS @ NOP,New Orleans,home_city_assumption
9,0021401219,2015-04-15,UTA,UTA @ HOU,Houston,home_city_assumption


# 1.4.2 Identify Games with Missing City Assignments

In [5]:
missing_city_games = games_with_city.loc[
    games_with_city["GAME_CITY"].isna(),
    [
        "GAME_ID",
        "SEASON",
        "GAME_DATE",
        "TEAM_ID",
        "TEAM_ABBREVIATION",
        "MATCHUP",
        "GAME_CITY"
    ]
].sort_values(["GAME_DATE", "GAME_ID"])

print(f"Missing team-game records: {len(missing_city_games)}")
print(f"Unique affected games: {missing_city_games['GAME_ID'].nunique()}")

display(missing_city_games)


Missing team-game records: 10
Unique affected games: 5


,GAME_ID,SEASON,GAME_DATE,TEAM_ID,TEAM_ABBREVIATION,MATCHUP,GAME_CITY
26250,0022400147,2024-25,2024-11-02,1610612748,MIA,MIA @ WAS,NaN
26251,0022400147,2024-25,2024-11-02,1610612764,WAS,MIA @ WAS,NaN
25666,0022401229,2024-25,2024-12-14,1610612749,MIL,ATL @ MIL,NaN
25669,0022401229,2024-25,2024-12-14,1610612737,ATL,ATL @ MIL,NaN
25667,0022401230,2024-25,2024-12-14,1610612760,OKC,HOU @ OKC,NaN
25668,0022401230,2024-25,2024-12-14,1610612745,HOU,HOU @ OKC,NaN
25111,0022400621,2024-25,2025-01-23,1610612759,SAS,SAS @ IND,NaN
25116,0022400621,2024-25,2025-01-23,1610612754,IND,SAS @ IND,NaN
25085,0022400633,2024-25,2025-01-25,1610612759,SAS,IND @ SAS,NaN
25094,0022400633,2024-25,2025-01-25,1610612754,IND,IND @ SAS,NaN


# 1.4.3 Resolve Missing Game Cities

In [6]:
verified_locations = {
    "0022400147": ("Mexico City", "Mexico City", "Mexico"),
    "0022401229": ("Las Vegas", "Nevada", "USA"),
    "0022401230": ("Las Vegas", "Nevada", "USA"),
    "0022400621": ("Paris", "Île-de-France", "France"),
    "0022400633": ("Paris", "Île-de-France", "France")
}

for game_id, (city, state, country) in verified_locations.items():
    mask = games_with_city["GAME_ID"] == game_id
    games_with_city.loc[mask, "GAME_CITY"] = city
    games_with_city.loc[mask, "GAME_STATE_PROVINCE"] = state
    games_with_city.loc[mask, "GAME_COUNTRY"] = country
    games_with_city.loc[mask, "LOCATION_STATUS"] = "verified_special_location"

print(f"Missing game cities: {games_with_city['GAME_CITY'].isna().sum()}")
print(f"Verified special games: {len(verified_locations)}")

display(
    games_with_city.loc[
        games_with_city["GAME_ID"].isin(verified_locations),
        ["GAME_ID", "GAME_DATE", "MATCHUP", "GAME_CITY", "GAME_COUNTRY"]
    ].drop_duplicates("GAME_ID").sort_values("GAME_DATE")
)


Missing game cities: 0
Verified special games: 5


,GAME_ID,GAME_DATE,MATCHUP,GAME_CITY,GAME_COUNTRY
26250,0022400147,2024-11-02,MIA @ WAS,Mexico City,Mexico
25666,0022401229,2024-12-14,ATL @ MIL,Las Vegas,USA
25667,0022401230,2024-12-14,HOU @ OKC,Las Vegas,USA
25111,0022400621,2025-01-23,SAS @ IND,Paris,France
25085,0022400633,2025-01-25,IND @ SAS,Paris,France


# 1.4.4 Handle Historical Special Game Locations

In [7]:
games_with_city["GAME_DATE"] = pd.to_datetime(games_with_city["GAME_DATE"])

bubble_mask = (
    (games_with_city["SEASON"] == "2019-20")
    & games_with_city["GAME_DATE"].between("2020-07-30", "2020-08-14")
)

raptors_home_ids = games_with_city.loc[
    (games_with_city["SEASON"] == "2020-21")
    & (games_with_city["TEAM_ABBREVIATION"] == "TOR")
    & (games_with_city["MATCHUP"].str.contains("vs.", regex=False)),
    "GAME_ID"
].unique()

raptors_mask = games_with_city["GAME_ID"].isin(raptors_home_ids)

games_with_city.loc[
    bubble_mask,
    ["GAME_CITY", "GAME_STATE_PROVINCE", "GAME_COUNTRY", "LOCATION_STATUS"]
] = ["Orlando", "Florida", "USA", "bubble_override"]

games_with_city.loc[
    raptors_mask,
    ["GAME_CITY", "GAME_STATE_PROVINCE", "GAME_COUNTRY", "LOCATION_STATUS"]
] = ["Tampa", "Florida", "USA", "temporary_home_override"]

print(f"Bubble games: {games_with_city.loc[bubble_mask, 'GAME_ID'].nunique()}")
print(f"Raptors Tampa home games: {len(raptors_home_ids)}")
print(f"Missing game cities: {games_with_city['GAME_CITY'].isna().sum()}")

display(
    games_with_city.loc[
        bubble_mask | raptors_mask,
        ["SEASON", "GAME_DATE", "MATCHUP", "GAME_CITY", "LOCATION_STATUS"]
    ].drop_duplicates().sort_values("GAME_DATE").head(15)
)


Bubble games: 88
Raptors Tampa home games: 36
Missing game cities: 0


,SEASON,GAME_DATE,MATCHUP,GAME_CITY,LOCATION_STATUS
12475,2019-20,2020-07-30,LAC @ LAL,Orlando,bubble_override
12474,2019-20,2020-07-30,UTA @ NOP,Orlando,bubble_override
12473,2019-20,2020-07-30,LAL vs. LAC,Orlando,bubble_override
12472,2019-20,2020-07-30,NOP vs. UTA,Orlando,bubble_override
12467,2019-20,2020-07-31,POR vs. MEM,Orlando,bubble_override
12468,2019-20,2020-07-31,MIL vs. BOS,Orlando,bubble_override
12460,2019-20,2020-07-31,ORL @ BKN,Orlando,bubble_override
12461,2019-20,2020-07-31,SAC @ SAS,Orlando,bubble_override
12462,2019-20,2020-07-31,SAS vs. SAC,Orlando,bubble_override
12463,2019-20,2020-07-31,HOU @ DAL,Orlando,bubble_override


# 1.4.5 Apply Verified International Game Locations

In [8]:
verified_games = [
    ("2014-11-12", "HOU", "MIN", "Mexico City", "Mexico City", "Mexico"),
    ("2015-12-03", "BOS", "SAC", "Mexico City", "Mexico City", "Mexico"),
    ("2017-01-12", "DAL", "PHX", "Mexico City", "Mexico City", "Mexico"),
    ("2017-01-14", "PHX", "SAS", "Mexico City", "Mexico City", "Mexico"),
    ("2017-12-07", "BKN", "OKC", "Mexico City", "Mexico City", "Mexico"),
    ("2017-12-09", "MIA", "BKN", "Mexico City", "Mexico City", "Mexico"),
    ("2018-12-13", "ORL", "CHI", "Mexico City", "Mexico City", "Mexico"),
    ("2018-12-15", "ORL", "UTA", "Mexico City", "Mexico City", "Mexico"),
    ("2019-12-12", "DAL", "DET", "Mexico City", "Mexico City", "Mexico"),
    ("2019-12-14", "SAS", "PHX", "Mexico City", "Mexico City", "Mexico"),
    ("2022-12-17", "MIA", "SAS", "Mexico City", "Mexico City", "Mexico"),
    ("2023-11-09", "ATL", "ORL", "Mexico City", "Mexico City", "Mexico"),
    ("2024-11-02", "MIA", "WAS", "Mexico City", "Mexico City", "Mexico"),
    ("2015-01-15", "MIL", "NYK", "London", "England", "UK"),
    ("2016-01-14", "TOR", "ORL", "London", "England", "UK"),
    ("2017-01-12", "DEN", "IND", "London", "England", "UK"),
    ("2018-01-11", "BOS", "PHI", "London", "England", "UK"),
    ("2019-01-17", "WAS", "NYK", "London", "England", "UK"),
    ("2020-01-24", "MIL", "CHA", "Paris", "Île-de-France", "France"),
    ("2023-01-19", "CHI", "DET", "Paris", "Île-de-France", "France"),
    ("2024-01-11", "BKN", "CLE", "Paris", "Île-de-France", "France"),
    ("2025-01-23", "SAS", "IND", "Paris", "Île-de-France", "France"),
    ("2025-01-25", "SAS", "IND", "Paris", "Île-de-France", "France")
]

game_pairs = (
    games_with_city.groupby(["GAME_ID", "GAME_DATE"])["TEAM_ABBREVIATION"]
    .agg(lambda teams: frozenset(teams))
    .reset_index(name="TEAM_PAIR")
)

verified_matches = []
unmatched_games = []

for date, team_a, team_b, city, state, country in verified_games:
    matches = game_pairs.loc[
        (game_pairs["GAME_DATE"] == pd.Timestamp(date))
        & (game_pairs["TEAM_PAIR"] == frozenset([team_a, team_b]))
    ]

    if len(matches) == 1:
        game_id = matches.iloc[0]["GAME_ID"]
        mask = games_with_city["GAME_ID"] == game_id

        games_with_city.loc[
            mask,
            ["GAME_CITY", "GAME_STATE_PROVINCE", "GAME_COUNTRY", "LOCATION_STATUS"]
        ] = [city, state, country, "verified_international_game"]

        verified_matches.append((game_id, date, team_a, team_b, city))
    else:
        unmatched_games.append((date, team_a, team_b))

verified_international = pd.DataFrame(
    verified_matches,
    columns=["GAME_ID", "GAME_DATE", "TEAM_A", "TEAM_B", "GAME_CITY"]
)

print(f"International games matched: {len(verified_international)}")
print(f"Unmatched games: {len(unmatched_games)}")
print(f"Missing game cities: {games_with_city['GAME_CITY'].isna().sum()}")

display(verified_international.sort_values("GAME_DATE"))

if unmatched_games:
    display(
        pd.DataFrame(
            unmatched_games,
            columns=["GAME_DATE", "TEAM_A", "TEAM_B"]
        )
    )


International games matched: 23
Unmatched games: 0
Missing game cities: 0


,GAME_ID,GAME_DATE,TEAM_A,TEAM_B,GAME_CITY
0,0021400117,2014-11-12,HOU,MIN,Mexico City
13,0021400585,2015-01-15,MIL,NYK,London
1,0021500282,2015-12-03,BOS,SAC,Mexico City
14,0021500588,2016-01-14,TOR,ORL,London
2,0021600591,2017-01-12,DAL,PHX,Mexico City
15,0021600589,2017-01-12,DEN,IND,London
3,0021600605,2017-01-14,PHX,SAS,Mexico City
4,0021700365,2017-12-07,BKN,OKC,Mexico City
5,0021700376,2017-12-09,MIA,BKN,Mexico City
16,0021700614,2018-01-11,BOS,PHI,London


# 1.4.6 Apply Verified Domestic Neutral-Site Locations

In [9]:
verified_domestic_games = [
    ("2023-12-07", "IND", "MIL", "Las Vegas", "Nevada", "USA"),
    ("2023-12-07", "LAL", "NOP", "Las Vegas", "Nevada", "USA"),
    ("2024-12-14", "ATL", "MIL", "Las Vegas", "Nevada", "USA"),
    ("2024-12-14", "HOU", "OKC", "Las Vegas", "Nevada", "USA"),
    ("2023-04-06", "SAS", "POR", "Austin", "Texas", "USA"),
    ("2023-04-08", "SAS", "MIN", "Austin", "Texas", "USA"),
    ("2024-03-15", "SAS", "DEN", "Austin", "Texas", "USA"),
    ("2024-03-17", "SAS", "BKN", "Austin", "Texas", "USA"),
    ("2025-02-20", "SAS", "PHX", "Austin", "Texas", "USA"),
    ("2025-02-21", "SAS", "DET", "Austin", "Texas", "USA")
]

domestic_matches = []
domestic_unmatched = []

for date, team_a, team_b, city, state, country in verified_domestic_games:
    matches = game_pairs.loc[
        (game_pairs["GAME_DATE"] == pd.Timestamp(date))
        & (game_pairs["TEAM_PAIR"] == frozenset([team_a, team_b]))
    ]

    if len(matches) == 1:
        game_id = matches.iloc[0]["GAME_ID"]
        mask = games_with_city["GAME_ID"] == game_id

        games_with_city.loc[
            mask,
            ["GAME_CITY", "GAME_STATE_PROVINCE", "GAME_COUNTRY", "LOCATION_STATUS"]
        ] = [city, state, country, "verified_domestic_special_game"]

        domestic_matches.append((game_id, date, team_a, team_b, city))
    else:
        domestic_unmatched.append((date, team_a, team_b))

verified_domestic = pd.DataFrame(
    domestic_matches,
    columns=["GAME_ID", "GAME_DATE", "TEAM_A", "TEAM_B", "GAME_CITY"]
)

print(f"Domestic special games matched: {len(verified_domestic)}")
print(f"Unmatched games: {len(domestic_unmatched)}")
print(f"Missing game cities: {games_with_city['GAME_CITY'].isna().sum()}")

display(verified_domestic.sort_values("GAME_DATE"))

if domestic_unmatched:
    display(
        pd.DataFrame(
            domestic_unmatched,
            columns=["GAME_DATE", "TEAM_A", "TEAM_B"]
        )
    )


Domestic special games matched: 10
Unmatched games: 0
Missing game cities: 0


,GAME_ID,GAME_DATE,TEAM_A,TEAM_B,GAME_CITY
4,0022201199,2023-04-06,SAS,POR,Austin
5,0022201214,2023-04-08,SAS,MIN,Austin
0,0022301229,2023-12-07,IND,MIL,Las Vegas
1,0022301230,2023-12-07,LAL,NOP,Las Vegas
6,0022300965,2024-03-15,SAS,DEN,Austin
7,0022300981,2024-03-17,SAS,BKN,Austin
2,0022401229,2024-12-14,ATL,MIL,Las Vegas
3,0022401230,2024-12-14,HOU,OKC,Las Vegas
8,0022400795,2025-02-20,SAS,PHX,Austin
9,0022400802,2025-02-21,SAS,DET,Austin


# 1.4.7 Validate and Save Game Locations

In [10]:
location_consistency = (
    games_with_city.groupby("GAME_ID")
    .agg(
        team_records=("TEAM_ID", "size"),
        city_count=("GAME_CITY", "nunique"),
        country_count=("GAME_COUNTRY", "nunique")
    )
    .reset_index()
)

invalid_games = location_consistency.loc[
    (location_consistency["team_records"] != 2)
    | (location_consistency["city_count"] != 1)
    | (location_consistency["country_count"] != 1)
]

location_summary = (
    games_with_city.drop_duplicates("GAME_ID")
    .groupby("LOCATION_STATUS")
    .size()
    .reset_index(name="GAMES")
    .sort_values("GAMES", ascending=False)
)

output_path = Path("../data/processed/nba_games_with_locations.csv")

games_with_city.to_csv(output_path, index=False)

print(f"Total games: {games_with_city['GAME_ID'].nunique():,}")
print(f"Missing cities: {games_with_city['GAME_CITY'].isna().sum()}")
print(f"Inconsistent games: {len(invalid_games)}")
print(f"Saved: {output_path}")

display(location_summary)

if len(invalid_games) > 0:
    display(invalid_games)


Total games: 13,209
Missing cities: 0
Inconsistent games: 0
Saved: ../data/processed/nba_games_with_locations.csv


,LOCATION_STATUS,GAMES
1,home_city_assumption,13052
0,bubble_override,88
2,temporary_home_override,36
4,verified_international_game,23
3,verified_domestic_special_game,10


# 1.5.1 Identify Unique Game Locations

In [11]:
unique_locations = (
    games_with_city[
        ["GAME_CITY", "GAME_STATE_PROVINCE", "GAME_COUNTRY"]
    ]
    .drop_duplicates()
    .sort_values(["GAME_COUNTRY", "GAME_STATE_PROVINCE", "GAME_CITY"])
    .reset_index(drop=True)
)

print(f"Unique game locations: {len(unique_locations)}")

display(unique_locations)


Unique game locations: 36


,GAME_CITY,GAME_STATE_PROVINCE,GAME_COUNTRY
0,Toronto,Ontario,Canada
1,Paris,Île-de-France,France
2,Mexico City,Mexico City,Mexico
3,London,England,UK
4,Phoenix,Arizona,USA
5,Los Angeles,California,USA
6,Oakland,California,USA
7,Sacramento,California,USA
8,San Francisco,California,USA
9,Denver,Colorado,USA


# 1.5.2 Assign City Coordinates

In [12]:
city_coordinates = {
    "Toronto": (43.6532, -79.3832),
    "Paris": (48.8566, 2.3522),
    "Mexico City": (19.4326, -99.1332),
    "London": (51.5074, -0.1278),
    "Phoenix": (33.4484, -112.0740),
    "Los Angeles": (34.0522, -118.2437),
    "Oakland": (37.8044, -122.2712),
    "Sacramento": (38.5816, -121.4944),
    "San Francisco": (37.7749, -122.4194),
    "Denver": (39.7392, -104.9903),
    "Washington": (38.9072, -77.0369),
    "Miami": (25.7617, -80.1918),
    "Orlando": (28.5383, -81.3792),
    "Tampa": (27.9506, -82.4572),
    "Atlanta": (33.7490, -84.3880),
    "Chicago": (41.8781, -87.6298),
    "Indianapolis": (39.7684, -86.1581),
    "New Orleans": (29.9511, -90.0715),
    "Boston": (42.3601, -71.0589),
    "Detroit": (42.3314, -83.0458),
    "Minneapolis": (44.9778, -93.2650),
    "Las Vegas": (36.1699, -115.1398),
    "Brooklyn": (40.6782, -73.9442),
    "New York": (40.7128, -74.0060),
    "Charlotte": (35.2271, -80.8431),
    "Cleveland": (41.4993, -81.6944),
    "Oklahoma City": (35.4676, -97.5164),
    "Portland": (45.5152, -122.6784),
    "Philadelphia": (39.9526, -75.1652),
    "Memphis": (35.1495, -90.0490),
    "Austin": (30.2672, -97.7431),
    "Dallas": (32.7767, -96.7970),
    "Houston": (29.7604, -95.3698),
    "San Antonio": (29.4241, -98.4936),
    "Salt Lake City": (40.7608, -111.8910),
    "Milwaukee": (43.0389, -87.9065)
}

unique_locations["LATITUDE"] = unique_locations["GAME_CITY"].map(
    lambda city: city_coordinates.get(city, (None, None))[0]
)

unique_locations["LONGITUDE"] = unique_locations["GAME_CITY"].map(
    lambda city: city_coordinates.get(city, (None, None))[1]
)

print(f"Total cities: {len(unique_locations)}")
print(f"Missing coordinates: {unique_locations['LATITUDE'].isna().sum()}")

display(unique_locations)


Total cities: 36
Missing coordinates: 0


,GAME_CITY,GAME_STATE_PROVINCE,GAME_COUNTRY,LATITUDE,LONGITUDE
0,Toronto,Ontario,Canada,43.6532,-79.3832
1,Paris,Île-de-France,France,48.8566,2.3522
2,Mexico City,Mexico City,Mexico,19.4326,-99.1332
3,London,England,UK,51.5074,-0.1278
4,Phoenix,Arizona,USA,33.4484,-112.0740
5,Los Angeles,California,USA,34.0522,-118.2437
6,Oakland,California,USA,37.8044,-122.2712
7,Sacramento,California,USA,38.5816,-121.4944
8,San Francisco,California,USA,37.7749,-122.4194
9,Denver,Colorado,USA,39.7392,-104.9903


# 1.5.3 Calculate Team Travel Distances

In [13]:
import numpy as np

games_with_coordinates = games_with_city.merge(
    unique_locations[
        ["GAME_CITY", "GAME_STATE_PROVINCE", "GAME_COUNTRY", "LATITUDE", "LONGITUDE"]
    ],
    on=["GAME_CITY", "GAME_STATE_PROVINCE", "GAME_COUNTRY"],
    how="left",
    validate="many_to_one"
)

games_with_coordinates = games_with_coordinates.sort_values(
    ["TEAM_ID", "SEASON", "GAME_DATE", "GAME_ID"]
).reset_index(drop=True)

team_seasons = games_with_coordinates.groupby(["TEAM_ID", "SEASON"])

games_with_coordinates["PREVIOUS_CITY"] = team_seasons["GAME_CITY"].shift(1)
games_with_coordinates["PREVIOUS_LATITUDE"] = team_seasons["LATITUDE"].shift(1)
games_with_coordinates["PREVIOUS_LONGITUDE"] = team_seasons["LONGITUDE"].shift(1)

lat1 = np.radians(games_with_coordinates["PREVIOUS_LATITUDE"])
lon1 = np.radians(games_with_coordinates["PREVIOUS_LONGITUDE"])
lat2 = np.radians(games_with_coordinates["LATITUDE"])
lon2 = np.radians(games_with_coordinates["LONGITUDE"])

a = (
    np.sin((lat2 - lat1) / 2) ** 2
    + np.cos(lat1) * np.cos(lat2)
    * np.sin((lon2 - lon1) / 2) ** 2
)

games_with_coordinates["TRAVEL_DISTANCE_KM"] = (
    6371.0088 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
)

print(f"Total team-game records: {len(games_with_coordinates):,}")
print(f"Missing coordinates: {games_with_coordinates['LATITUDE'].isna().sum()}")
print(f"First games without previous travel: {games_with_coordinates['TRAVEL_DISTANCE_KM'].isna().sum()}")

display(
    games_with_coordinates[
        [
            "SEASON",
            "GAME_DATE",
            "TEAM_ABBREVIATION",
            "PREVIOUS_CITY",
            "GAME_CITY",
            "TRAVEL_DISTANCE_KM"
        ]
    ].head(15)
)


Total team-game records: 26,418
Missing coordinates: 0
First games without previous travel: 330


,SEASON,GAME_DATE,TEAM_ABBREVIATION,PREVIOUS_CITY,GAME_CITY,TRAVEL_DISTANCE_KM
0,2014-15,2014-10-29,ATL,NaN,Toronto,NaN
1,2014-15,2014-11-01,ATL,Toronto,Atlanta,1183.241109
2,2014-15,2014-11-05,ATL,Atlanta,San Antonio,1418.503166
3,2014-15,2014-11-07,ATL,San Antonio,Charlotte,1776.370443
4,2014-15,2014-11-08,ATL,Charlotte,Atlanta,364.069223
5,2014-15,2014-11-10,ATL,Atlanta,New York,1200.318456
6,2014-15,2014-11-12,ATL,New York,Atlanta,1200.318456
7,2014-15,2014-11-14,ATL,Atlanta,Atlanta,0.000000
8,2014-15,2014-11-15,ATL,Atlanta,Cleveland,893.713685
9,2014-15,2014-11-18,ATL,Cleveland,Atlanta,893.713685


# 1.6.1 Calculate Rest Days and Back-to-Back Games

In [14]:
games_with_coordinates = games_with_coordinates.sort_values(
    ["TEAM_ID", "SEASON", "GAME_DATE", "GAME_ID"]
).reset_index(drop=True)

team_seasons = games_with_coordinates.groupby(["TEAM_ID", "SEASON"])

games_with_coordinates["PREVIOUS_GAME_DATE"] = (
    team_seasons["GAME_DATE"].shift(1)
)

games_with_coordinates["REST_DAYS"] = (
    games_with_coordinates["GAME_DATE"]
    - games_with_coordinates["PREVIOUS_GAME_DATE"]
).dt.days - 1

games_with_coordinates["IS_B2B"] = (
    games_with_coordinates["REST_DAYS"] == 0
).astype(int)

print(f"Total records: {len(games_with_coordinates):,}")
print(f"Missing rest days: {games_with_coordinates['REST_DAYS'].isna().sum()}")
print(f"Back-to-back games: {games_with_coordinates['IS_B2B'].sum():,}")

display(
    games_with_coordinates[
        [
            "SEASON",
            "GAME_DATE",
            "TEAM_ABBREVIATION",
            "PREVIOUS_GAME_DATE",
            "REST_DAYS",
            "IS_B2B"
        ]
    ].head(15)
)


Total records: 26,418
Missing rest days: 330
Back-to-back games: 4,904


,SEASON,GAME_DATE,TEAM_ABBREVIATION,PREVIOUS_GAME_DATE,REST_DAYS,IS_B2B
0,2014-15,2014-10-29,ATL,NaT,NaN,0
1,2014-15,2014-11-01,ATL,2014-10-29,2.0,0
2,2014-15,2014-11-05,ATL,2014-11-01,3.0,0
3,2014-15,2014-11-07,ATL,2014-11-05,1.0,0
4,2014-15,2014-11-08,ATL,2014-11-07,0.0,1
5,2014-15,2014-11-10,ATL,2014-11-08,1.0,0
6,2014-15,2014-11-12,ATL,2014-11-10,1.0,0
7,2014-15,2014-11-14,ATL,2014-11-12,1.0,0
8,2014-15,2014-11-15,ATL,2014-11-14,0.0,1
9,2014-15,2014-11-18,ATL,2014-11-15,2.0,0


# 1.6.2 Calculate Schedule Density Features

In [15]:
games_with_coordinates = games_with_coordinates.sort_values(
    ["TEAM_ID", "SEASON", "GAME_DATE", "GAME_ID"]
).reset_index(drop=True)

team_seasons = games_with_coordinates.groupby(["TEAM_ID", "SEASON"])

games_with_coordinates["IS_3IN4"] = (
    (
        games_with_coordinates["GAME_DATE"]
        - team_seasons["GAME_DATE"].shift(2)
    ).dt.days <= 3
).astype(int)

games_with_coordinates["GAMES_LAST_7D"] = 0

for _, group in games_with_coordinates.groupby(["TEAM_ID", "SEASON"]):
    dates = group["GAME_DATE"].to_numpy(dtype="datetime64[ns]")

    window_start = np.searchsorted(
        dates,
        dates - np.timedelta64(7, "D"),
        side = "left"
    )

    games_with_coordinates.loc[
        group.index, "GAMES_LAST_7D"
    ] = np.arange(len(group)) - window_start

print(f"Total records: {len(games_with_coordinates):,}")
print(f"Three games in four days: {games_with_coordinates['IS_3IN4'].sum():,}")
print(f"Average games in last 7 days: {games_with_coordinates['GAMES_LAST_7D'].mean():.2f}")

display(
    games_with_coordinates.loc[
        (games_with_coordinates["TEAM_ABBREVIATION"] == "ATL")
        & (games_with_coordinates["SEASON"] == "2014-15"),
        [
            "GAME_DATE",
            "REST_DAYS",
            "IS_B2B",
            "IS_3IN4",
            "GAMES_LAST_7D"
        ]
    ].head(15)
)


Total records: 26,418
Three games in four days: 7,086
Average games in last 7 days: 3.06


,GAME_DATE,REST_DAYS,IS_B2B,IS_3IN4,GAMES_LAST_7D
0,2014-10-29,NaN,0,0,0
1,2014-11-01,2.0,0,0,1
2,2014-11-05,3.0,0,0,2
3,2014-11-07,1.0,0,0,2
4,2014-11-08,0.0,1,1,3
5,2014-11-10,1.0,0,1,3
6,2014-11-12,1.0,0,0,4
7,2014-11-14,1.0,0,0,4
8,2014-11-15,0.0,1,1,4
9,2014-11-18,2.0,0,0,3


## 1.6.3 Calculate Consecutive Road Games

In [16]:
verified_home_teams = {
    "0022400147": "WAS",
    "0022401229": "MIL",
    "0022401230": "OKC",
    "0022400621": "IND",
    "0022400633": "SAS"
}

game_ids = games_with_coordinates["GAME_ID"].astype(str).str.zfill(10)

for game_id, home_team in verified_home_teams.items():
    mask = game_ids == game_id

    games_with_coordinates.loc[mask, "is_home"] = (
        games_with_coordinates.loc[mask, "TEAM_ABBREVIATION"]
        == home_team
    )

games_with_coordinates = games_with_coordinates.sort_values(
    ["TEAM_ID", "SEASON", "GAME_DATE", "GAME_ID"]
).reset_index(drop=True)

road_games = ~games_with_coordinates["is_home"].astype(bool)

home_streak_groups = (
    (~road_games)
    .groupby([
        games_with_coordinates["TEAM_ID"],
        games_with_coordinates["SEASON"]
    ])
    .cumsum()
)

games_with_coordinates["CONSECUTIVE_ROAD_GAMES"] = (
    road_games.astype(int)
    .groupby([
        games_with_coordinates["TEAM_ID"],
        games_with_coordinates["SEASON"],
        home_streak_groups
    ])
    .cumsum()
)

print(
    "Home games with nonzero road streak:",
    (
        games_with_coordinates.loc[
            games_with_coordinates["is_home"],
            "CONSECUTIVE_ROAD_GAMES"
        ] != 0
    ).sum()
)


Home games with nonzero road streak: 0


# 1.7.1 Correct Home Team Assignments and Construct Game-Level Dataset

In [17]:
verified_home_teams = {
    "0022400147": "WAS",
    "0022401229": "MIL",
    "0022401230": "OKC",
    "0022400621": "IND",
    "0022400633": "SAS"
}

for game_id, home_team in verified_home_teams.items():
    mask = games_with_coordinates["GAME_ID"] == game_id

    games_with_coordinates.loc[mask, "is_home"] = (
        games_with_coordinates.loc[mask, "TEAM_ABBREVIATION"] == home_team
    )

neutral_statuses = [
    "verified_international_game",
    "verified_domestic_special_game",
    "bubble_override"
]

games_with_coordinates["IS_NEUTRAL_SITE"] = (
    games_with_coordinates["LOCATION_STATUS"].isin(neutral_statuses)
).astype(int)

games_with_coordinates.loc[
    games_with_coordinates["GAME_ID"] == "0022400147",
    "IS_NEUTRAL_SITE"
] = 1

home_counts = games_with_coordinates.groupby("GAME_ID")["is_home"].sum()

assert home_counts.eq(1).all()
assert games_with_coordinates.groupby("GAME_ID").size().eq(2).all()

game_features = [
    "TEAM_ID",
    "TEAM_ABBREVIATION",
    "REST_DAYS",
    "IS_B2B",
    "IS_3IN4",
    "GAMES_LAST_7D",
    "CONSECUTIVE_ROAD_GAMES",
    "TRAVEL_DISTANCE_KM",
    "WL",
    "PTS"
]

home_games = games_with_coordinates.loc[
    games_with_coordinates["is_home"],
    ["GAME_ID", "SEASON", "GAME_DATE", "IS_NEUTRAL_SITE"] + game_features
].rename(columns={col: f"HOME_{col}" for col in game_features})

away_games = games_with_coordinates.loc[
    ~games_with_coordinates["is_home"],
    ["GAME_ID"] + game_features
].rename(columns={col: f"AWAY_{col}" for col in game_features})

game_level = home_games.merge(
    away_games,
    on="GAME_ID",
    how="inner",
    validate="one_to_one"
)

game_level["HOME_WIN"] = (game_level["HOME_WL"] == "W").astype(int)

game_level["POINT_DIFF"] = (
    game_level["HOME_PTS"] - game_level["AWAY_PTS"]
)

print(f"Total game-level records: {len(game_level):,}")
print(f"Unique games: {game_level['GAME_ID'].nunique():,}")
print(f"Missing game outcomes: {game_level['HOME_WL'].isna().sum():,}")
print(f"Flagged neutral-site games: {game_level['IS_NEUTRAL_SITE'].sum():,}")

display(
    game_level[
        [
            "GAME_ID", "GAME_DATE",
            "HOME_TEAM_ABBREVIATION",
            "AWAY_TEAM_ABBREVIATION",
            "IS_NEUTRAL_SITE", "POINT_DIFF", "HOME_WIN"
        ]
    ].head(10)
)


Total game-level records: 13,209
Unique games: 13,209
Missing game outcomes: 0
Flagged neutral-site games: 121


,GAME_ID,GAME_DATE,HOME_TEAM_ABBREVIATION,AWAY_TEAM_ABBREVIATION,IS_NEUTRAL_SITE,POINT_DIFF,HOME_WIN
0,0021400032,2014-11-01,ATL,IND,0,10,1
1,0021400084,2014-11-08,ATL,NYK,0,7,1
2,0021400110,2014-11-12,ATL,UTA,0,3,1
3,0021400124,2014-11-14,ATL,MIA,0,11,1
4,0021400155,2014-11-18,ATL,LAL,0,-5,0
5,0021400174,2014-11-21,ATL,DET,0,10,1
6,0021400215,2014-11-26,ATL,TOR,0,-11,0
7,0021400225,2014-11-28,ATL,NOP,0,9,1
8,0021400237,2014-11-29,ATL,CHA,0,30,1
9,0021400254,2014-12-02,ATL,BOS,0,4,1


# 1.7.2 Validate Game-Level Features

In [18]:
feature_columns = [
    "HOME_REST_DAYS",
    "AWAY_REST_DAYS",
    "HOME_IS_B2B",
    "AWAY_IS_B2B",
    "HOME_IS_3IN4",
    "AWAY_IS_3IN4",
    "HOME_GAMES_LAST_7D",
    "AWAY_GAMES_LAST_7D",
    "HOME_TRAVEL_DISTANCE_KM",
    "AWAY_TRAVEL_DISTANCE_KM",
    "IS_NEUTRAL_SITE"
]

missing_summary = (
    game_level[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

validation_summary = pd.Series({
    "Total games": len(game_level),
    "Duplicate game IDs": game_level["GAME_ID"].duplicated().sum(),
    "Missing targets": game_level["HOME_WIN"].isna().sum(),
    "Inconsistent win/loss labels": (
        game_level["HOME_WL"] == game_level["AWAY_WL"]
    ).sum(),
    "Inconsistent scores": (
        (game_level["HOME_PTS"] > game_level["AWAY_PTS"]).astype(int)
        != game_level["HOME_WIN"]
    ).sum(),
    "Negative rest days": (
        (game_level["HOME_REST_DAYS"] < 0)
        | (game_level["AWAY_REST_DAYS"] < 0)
    ).sum(),
    "Negative travel distances": (
        (game_level["HOME_TRAVEL_DISTANCE_KM"] < 0)
        | (game_level["AWAY_TRAVEL_DISTANCE_KM"] < 0)
    ).sum()
})

print("Validation Summary")
display(validation_summary.to_frame("Value"))

print("Missing Feature Values")
display(missing_summary.to_frame("Missing Count"))

print("Feature Statistics")
display(game_level[feature_columns].describe().round(2))


Validation Summary


,Value
Total games,13209
Duplicate game IDs,0
Missing targets,0
Inconsistent win/loss labels,0
Inconsistent scores,0
Negative rest days,0
Negative travel distances,0


Missing Feature Values


,Missing Count
HOME_REST_DAYS,167
HOME_TRAVEL_DISTANCE_KM,167
AWAY_REST_DAYS,163
AWAY_TRAVEL_DISTANCE_KM,163
HOME_IS_B2B,0
AWAY_IS_B2B,0
HOME_IS_3IN4,0
HOME_GAMES_LAST_7D,0
AWAY_IS_3IN4,0
AWAY_GAMES_LAST_7D,0


Feature Statistics


,HOME_REST_DAYS,AWAY_REST_DAYS,HOME_IS_B2B,AWAY_IS_B2B,HOME_IS_3IN4,AWAY_IS_3IN4,HOME_GAMES_LAST_7D,AWAY_GAMES_LAST_7D,HOME_TRAVEL_DISTANCE_KM,AWAY_TRAVEL_DISTANCE_KM,IS_NEUTRAL_SITE
count,13042.00,13046.00,13209.00,13209.00,13209.00,13209.00,13209.00,13209.00,13042.00,13046.00,13209.00
mean,1.31,1.15,0.14,0.23,0.25,0.29,3.05,3.07,613.37,1140.31,0.01
std,4.40,4.03,0.35,0.42,0.43,0.45,0.88,0.87,906.97,800.87,0.10
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,1.00,1.00,0.00,0.00,0.00,0.00,3.00,3.00,0.00,576.28,0.00
50%,1.00,1.00,0.00,0.00,0.00,0.00,3.00,3.00,0.00,975.88,0.00
75%,1.00,1.00,0.00,0.00,0.00,1.00,4.00,4.00,1060.09,1528.63,0.00
max,145.00,144.00,1.00,1.00,1.00,1.00,5.00,5.00,8317.82,7356.58,1.00


# 1.7.3 Inspect Extended Rest Periods

In [19]:
extended_rest = games_with_coordinates.loc[
    games_with_coordinates["REST_DAYS"] > 14,
    [
        "SEASON",
        "GAME_DATE",
        "TEAM_ABBREVIATION",
        "MATCHUP",
        "PREVIOUS_GAME_DATE",
        "REST_DAYS"
    ]
].sort_values("REST_DAYS", ascending=False)

print(f"Team-game records with rest periods over 14 days: {len(extended_rest):,}")

display(
    extended_rest.groupby("SEASON")
    .agg(
        records=("REST_DAYS", "size"),
        maximum_rest_days=("REST_DAYS", "max")
    )
    .reset_index()
)

display(extended_rest.head(20))


Team-game records with rest periods over 14 days: 22


,SEASON,records,maximum_rest_days
0,2019-20,22,145.0


,SEASON,GAME_DATE,TEAM_ABBREVIATION,MATCHUP,PREVIOUS_GAME_DATE,REST_DAYS
20730,2019-20,2020-08-01,OKC,OKC vs. UTA,2020-03-08,145.0
18967,2019-20,2020-07-31,SAC,SAC @ SAS,2020-03-08,144.0
21612,2019-20,2020-08-01,TOR,TOR vs. LAL,2020-03-09,144.0
11037,2019-20,2020-07-31,MIL,MIL vs. BOS,2020-03-09,143.0
3108,2019-20,2020-07-30,NOP,NOP vs. UTA,2020-03-08,143.0
15435,2019-20,2020-08-01,IND,IND vs. PHI,2020-03-10,143.0
19848,2019-20,2020-07-31,SAS,SAS vs. SAC,2020-03-10,142.0
17201,2019-20,2020-07-31,PHX,PHX @ WAS,2020-03-10,142.0
7508,2019-20,2020-07-31,HOU,HOU @ DAL,2020-03-10,142.0
5751,2019-20,2020-08-01,DEN,DEN vs. MIA,2020-03-11,142.0


# 1.7.4 Handle Extended Rest Periods

In [20]:
game_level["HOME_REST_DAYS_CAPPED"] = game_level["HOME_REST_DAYS"].clip(upper=14)
game_level["AWAY_REST_DAYS_CAPPED"] = game_level["AWAY_REST_DAYS"].clip(upper=14)

game_level["IS_COVID_RESTART"] = (
    (game_level["SEASON"] == "2019-20")
    & (
        (game_level["HOME_REST_DAYS"] > 14)
        | (game_level["AWAY_REST_DAYS"] > 14)
    )
).astype(int)

print(f"Total games: {len(game_level):,}")
print(f"COVID restart games: {game_level['IS_COVID_RESTART'].sum():,}")
print(f"Maximum capped home rest: {game_level['HOME_REST_DAYS_CAPPED'].max():.0f}")
print(f"Maximum capped away rest: {game_level['AWAY_REST_DAYS_CAPPED'].max():.0f}")

display(
    game_level.loc[
        game_level["IS_COVID_RESTART"] == 1,
        [
            "GAME_DATE",
            "HOME_TEAM_ABBREVIATION",
            "AWAY_TEAM_ABBREVIATION",
            "HOME_REST_DAYS",
            "AWAY_REST_DAYS",
            "IS_COVID_RESTART"
        ]
    ].head(10)
)


Total games: 13,209
COVID restart games: 12
Maximum capped home rest: 14
Maximum capped away rest: 14


,GAME_DATE,HOME_TEAM_ABBREVIATION,AWAY_TEAM_ABBREVIATION,HOME_REST_DAYS,AWAY_REST_DAYS,IS_COVID_RESTART
1557,2020-07-30,NOP,UTA,143.0,142.0,1
2438,2020-07-31,DAL,HOU,141.0,142.0,1
2879,2020-08-01,DEN,MIA,142.0,142.0,1
4640,2020-07-30,LAL,LAC,141.0,141.0,1
5522,2020-07-31,MIL,BOS,143.0,142.0,1
6402,2020-07-31,BKN,ORL,142.0,142.0,1
7721,2020-08-01,IND,PHI,143.0,142.0,1
9046,2020-07-31,POR,MEM,142.0,142.0,1
9925,2020-07-31,SAS,SAC,142.0,144.0,1
10367,2020-08-01,OKC,UTA,145.0,1.0,1


## 1.7.5 Final Data Validation

In [21]:
required_columns = [
    "GAME_ID",
    "SEASON",
    "GAME_DATE",
    "HOME_TEAM_ID",
    "AWAY_TEAM_ID",
    "HOME_PTS",
    "AWAY_PTS",
    "HOME_WL",
    "AWAY_WL",
    "HOME_WIN",
    "POINT_DIFF",
    "HOME_REST_DAYS",
    "AWAY_REST_DAYS",
    "HOME_IS_B2B",
    "AWAY_IS_B2B",
    "HOME_IS_3IN4",
    "AWAY_IS_3IN4",
    "HOME_GAMES_LAST_7D",
    "AWAY_GAMES_LAST_7D",
    "HOME_CONSECUTIVE_ROAD_GAMES",
    "AWAY_CONSECUTIVE_ROAD_GAMES",
    "HOME_TRAVEL_DISTANCE_KM",
    "AWAY_TRAVEL_DISTANCE_KM"
]

missing_columns = [
    col for col in required_columns
    if col not in game_level.columns
]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    checks = {}

    checks["Game count"] = len(game_level) == 13209

    checks["Unique game IDs"] = (
        game_level["GAME_ID"].is_unique
    )

    checks["Different teams"] = (
        game_level["HOME_TEAM_ID"] != game_level["AWAY_TEAM_ID"]
    ).all()

    checks["Valid win labels"] = (
        game_level["HOME_WIN"].isin([0, 1])
    ).all()

    checks["Opposite win/loss results"] = (
        (
            (game_level["HOME_WL"] == "W")
            & (game_level["AWAY_WL"] == "L")
        )
        |
        (
            (game_level["HOME_WL"] == "L")
            & (game_level["AWAY_WL"] == "W")
        )
    ).all()

    checks["Correct home win"] = (
        game_level["HOME_WIN"]
        == (game_level["HOME_WL"] == "W").astype(int)
    ).all()

    checks["Correct point differential"] = (
        game_level["POINT_DIFF"]
        == game_level["HOME_PTS"] - game_level["AWAY_PTS"]
    ).all()

    checks["Score and outcome consistency"] = (
        (game_level["POINT_DIFF"] > 0)
        == (game_level["HOME_WIN"] == 1)
    ).all()

    for side in ["HOME", "AWAY"]:
        rest = game_level[f"{side}_REST_DAYS"]
        b2b = game_level[f"{side}_IS_B2B"]
        travel = game_level[f"{side}_TRAVEL_DISTANCE_KM"]
        road = game_level[f"{side}_CONSECUTIVE_ROAD_GAMES"]
        games_7d = game_level[f"{side}_GAMES_LAST_7D"]
        games_3in4 = game_level[f"{side}_IS_3IN4"]

        checks[f"{side} nonnegative rest"] = (
            rest.dropna() >= 0
        ).all()

        checks[f"{side} correct B2B"] = (
            b2b == rest.eq(0).astype(int)
        ).all()

        checks[f"{side} nonnegative travel"] = (
            travel.dropna() >= 0
        ).all()

        checks[f"{side} valid road streak"] = (
            road.notna() & (road >= 0)
        ).all()

        checks[f"{side} valid 7-day count"] = (
            games_7d.notna()
            & games_7d.between(0, 6)
        ).all()

        checks[f"{side} valid 3-in-4 flag"] = (
            games_3in4.isin([0, 1])
        ).all()

    checks["Home road streak is zero"] = (
        game_level["HOME_CONSECUTIVE_ROAD_GAMES"] == 0
    ).all()

    validation_results = pd.DataFrame(
        [
            {
                "Check": name,
                "Status": "PASS" if result else "FAIL"
            }
            for name, result in checks.items()
        ]
    )

    display(validation_results)

    print("\nDataset shape:", game_level.shape)
    print("Unique games:", game_level["GAME_ID"].nunique())
    print("Passed checks:", sum(checks.values()), "/", len(checks))

    print("\nMissing values:")
    display(
        game_level[required_columns]
        .isna()
        .sum()
        .loc[lambda x: x > 0]
        .sort_values(ascending=False)
        .to_frame("Missing Count")
    )


,Check,Status
0,Game count,PASS
1,Unique game IDs,PASS
2,Different teams,PASS
3,Valid win labels,PASS
4,Opposite win/loss results,PASS
5,Correct home win,PASS
6,Correct point differential,PASS
7,Score and outcome consistency,PASS
8,HOME nonnegative rest,PASS
9,HOME correct B2B,PASS



Dataset shape: (13209, 29)
Unique games: 13209
Passed checks: 21 / 21

Missing values:


,Missing Count
HOME_REST_DAYS,167
HOME_TRAVEL_DISTANCE_KM,167
AWAY_REST_DAYS,163
AWAY_TRAVEL_DISTANCE_KM,163


# 1.8 Save Final Modeling Dataset

In [22]:
output_path = Path("../data/processed/nba_modeling_dataset.csv")

game_level = game_level.sort_values(
    ["GAME_DATE", "GAME_ID"]
).reset_index(drop=True)

assert len(game_level) == 13209
assert game_level["GAME_ID"].is_unique
assert game_level["HOME_WIN"].notna().all()

game_level.to_csv(output_path, index=False)

print(f"Saved dataset: {output_path}")
print(f"Total games: {len(game_level):,}")
print(f"Total columns: {len(game_level.columns)}")
print(f"Seasons: {game_level['SEASON'].nunique()}")
print(f"Date range: {game_level['GAME_DATE'].min().date()} to {game_level['GAME_DATE'].max().date()}")

display(game_level.head(5))


Saved dataset: ../data/processed/nba_modeling_dataset.csv
Total games: 13,209
Total columns: 29
Seasons: 11
Date range: 2014-10-28 to 2025-04-13


,GAME_ID,SEASON,GAME_DATE,IS_NEUTRAL_SITE,HOME_TEAM_ID,HOME_TEAM_ABBREVIATION,HOME_REST_DAYS,HOME_IS_B2B,HOME_IS_3IN4,HOME_GAMES_LAST_7D,...,AWAY_GAMES_LAST_7D,AWAY_CONSECUTIVE_ROAD_GAMES,AWAY_TRAVEL_DISTANCE_KM,AWAY_WL,AWAY_PTS,HOME_WIN,POINT_DIFF,HOME_REST_DAYS_CAPPED,AWAY_REST_DAYS_CAPPED,IS_COVID_RESTART
0,0021400001,2014-15,2014-10-28,0,1610612740,NOP,NaN,0,0,0,...,0,1,NaN,L,84,1,17,NaN,NaN,0
1,0021400002,2014-15,2014-10-28,0,1610612759,SAS,NaN,0,0,0,...,0,1,NaN,L,100,1,1,NaN,NaN,0
2,0021400003,2014-15,2014-10-28,0,1610612747,LAL,NaN,0,0,0,...,0,1,NaN,W,108,0,-18,NaN,NaN,0
3,0021400004,2014-15,2014-10-29,0,1610612766,CHA,NaN,0,0,0,...,0,1,NaN,L,106,1,2,NaN,NaN,0
4,0021400005,2014-15,2014-10-29,0,1610612754,IND,NaN,0,0,0,...,0,1,NaN,L,91,1,12,NaN,NaN,0
